# Toy Dataset

In [1]:
# Tạo dữ liệu 5 khách hàng
import pandas as pd
import numpy as np

data = pd.DataFrame({
    'Person': ['A', 'B', 'C', 'D', 'E'],
    'Feature_X': [0.8, 0.2, 0.9, 0.1, 0.5], # Ví dụ: Xác suất thích xe PKL (1=Rất thích)
    'W': [1, 0, 1, 0, 1],                  # Treatment: 1=Có mã, 0=Không
    'Y': [500, 100, 0, 150, 300]           # Outcome: Doanh thu thực tế
})

print("--- Dữ liệu gốc ---")
print(data)

--- Dữ liệu gốc ---
  Person  Feature_X  W    Y
0      A        0.8  1  500
1      B        0.2  0  100
2      C        0.9  1    0
3      D        0.1  0  150
4      E        0.5  1  300


# Stage 1

Step 1: Multi-output Gradient

In [6]:
# 1. Khởi tạo giá trị dự đoán ban đầu (Base score)
# Giả sử ta lấy trung bình của từng nhóm làm dự đoán khởi điểm
mean_y0 = data[data['W'] == 0]['Y'].mean() # Trung bình nhóm Control
mean_y1 = data[data['W'] == 1]['Y'].mean() # Trung bình nhóm Treatment

# Tạo vector dự đoán hiện tại [Y_hat(0), Y_hat(1)]
# Lúc đầu ai cũng được dự đoán giống nhau (Naive prediction)
data['Y_hat_0'] = mean_y0 
data['Y_hat_1'] = mean_y1

print("Init data: ")
print(data)

# 2. Tính Gradient và Hessian (Mấu chốt của bài báo)
def compute_masked_gradients(row):
    y_true = row['Y']
    y_pred_0 = row['Y_hat_0']
    y_pred_1 = row['Y_hat_1']
    w = row['W']
    
    # Loss function là MSE: L = 1/2 * (y_pred - y_true)^2
    # Gradient (g) = y_pred - y_true
    # Hessian (h) = 1 (với MSE)
    
    # LOGIC MASKING 
    if w == 0: # Nhóm Control
        # Tính cho nhóm 0
        g0 = y_pred_0 - y_true
        h0 = 1
        # MASK nhóm 1 (Gán bằng 0 vì không quan sát được)
        g1 = 0 
        h1 = 0
    else: # Nhóm Treatment (w=1)
        # MASK nhóm 0
        g0 = 0
        h0 = 0
        # Tính cho nhóm 1
        g1 = y_pred_1 - y_true
        h1 = 1
        
    return pd.Series([g0, h0, g1, h1], index=['g0', 'h0', 'g1', 'h1'])

# Áp dụng vào DataFrame
gradients = data.apply(compute_masked_gradients, axis=1)
data_stage1 = pd.concat([data, gradients], axis=1)

print("\n--- Kết quả tính Gradient (Stage 1) ---")
print(data)
print(data_stage1)

Init data: 
  Person  Feature_X  W    Y  Y_hat_0     Y_hat_1
0      A        0.8  1  500    125.0  266.666667
1      B        0.2  0  100    125.0  266.666667
2      C        0.9  1    0    125.0  266.666667
3      D        0.1  0  150    125.0  266.666667
4      E        0.5  1  300    125.0  266.666667

--- Kết quả tính Gradient (Stage 1) ---
  Person  Feature_X  W    Y  Y_hat_0     Y_hat_1
0      A        0.8  1  500    125.0  266.666667
1      B        0.2  0  100    125.0  266.666667
2      C        0.9  1    0    125.0  266.666667
3      D        0.1  0  150    125.0  266.666667
4      E        0.5  1  300    125.0  266.666667
  Person  Feature_X  W    Y  Y_hat_0     Y_hat_1    g0   h0          g1   h1
0      A        0.8  1  500    125.0  266.666667   0.0  0.0 -233.333333  1.0
1      B        0.2  0  100    125.0  266.666667  25.0  1.0    0.000000  0.0
2      C        0.9  1    0    125.0  266.666667   0.0  0.0  266.666667  1.0
3      D        0.1  0  150    125.0  266.666667 -2

Step 2: Newton step

In [7]:
# Tính tổng Gradient và Hessian cho cả nhóm
sum_g0 = data_stage1['g0'].sum()
sum_h0 = data_stage1['h0'].sum()

sum_g1 = data_stage1['g1'].sum()
sum_h1 = data_stage1['h1'].sum()

# Tính bước nhảy (Newton Step)
# Lưu ý: sum_h sẽ chính là số lượng mẫu thực tế quan sát được trong nhóm đó
delta_0 = - sum_g0 / (sum_h0 + 1e-5) # Cộng epsilon để tránh chia cho 0
delta_1 = - sum_g1 / (sum_h1 + 1e-5)

print("\n--- Cập nhật mô hình (Newton Step) ---")
print(f"Tổng lỗi nhóm Control (sum_g0): {sum_g0}, Số lượng mẫu (sum_h0): {sum_h0}")
print(f"--> Cần điều chỉnh Y_hat(0) một lượng: {delta_0:.2f}")

print(f"Tổng lỗi nhóm Treatment (sum_g1): {sum_g1}, Số lượng mẫu (sum_h1): {sum_h1}")
print(f"--> Cần điều chỉnh Y_hat(1) một lượng: {delta_1:.2f}")

# Cập nhật dự đoán (Đây là output của Stage 1 sau 1 vòng lặp)
data_stage1['Y_hat_0_new'] = data_stage1['Y_hat_0'] + delta_0
data_stage1['Y_hat_1_new'] = data_stage1['Y_hat_1'] + delta_1


--- Cập nhật mô hình (Newton Step) ---
Tổng lỗi nhóm Control (sum_g0): 0.0, Số lượng mẫu (sum_h0): 2.0
--> Cần điều chỉnh Y_hat(0) một lượng: -0.00
Tổng lỗi nhóm Treatment (sum_g1): 5.684341886080802e-14, Số lượng mẫu (sum_h1): 3.0
--> Cần điều chỉnh Y_hat(1) một lượng: -0.00


# Stage 2

In [8]:
def compute_surrogate_target(row):
    w = row['W']
    y_true = row['Y']
    y_pred_0 = row['Y_hat_0_new']
    y_pred_1 = row['Y_hat_1_new']
    
    if w == 1:
        # Nếu có mã (Treatment), Uplift = Thực tế - (Dự đoán nếu ko có mã)
        # Đây là phần "Gain" do mã mang lại
        return y_true - y_pred_0
    else:
        # Nếu ko mã (Control), Uplift = (Dự đoán nếu có mã) - Thực tế
        # Đây là phần "Tiếc nuối" (Potential Gain) nếu đã gửi mã
        return y_pred_1 - y_true

data_stage1['U_target'] = data_stage1.apply(compute_surrogate_target, axis=1)

print("\n--- Kết quả Stage 2: Surrogate Target (Nhãn Uplift giả lập) ---")
print(data_stage1[['Person', 'W', 'Y', 'Y_hat_0_new', 'Y_hat_1_new', 'U_target']])


--- Kết quả Stage 2: Surrogate Target (Nhãn Uplift giả lập) ---
  Person  W    Y  Y_hat_0_new  Y_hat_1_new    U_target
0      A  1  500        125.0   266.666667  375.000000
1      B  0  100        125.0   266.666667  166.666667
2      C  1    0        125.0   266.666667 -125.000000
3      D  0  150        125.0   266.666667  116.666667
4      E  1  300        125.0   266.666667  175.000000
